# L10 - Calcolo vettoriale con NumPy

Eseguire le celle in ordine con `Maiusc + Invio`.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np

## 1. Operazioni elemento per elemento

Gli operatori aritmetici tra array della stessa forma agiscono su ogni coppia di elementi corrispondenti. Tra un array e un numero, il numero viene applicato a ogni elemento.

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([10.0, 20.0, 30.0, 40.0])
print(a + b)
print(a * b)
print(a * 2)
print(a ** 2)
print(1 / a)

## 2. Funzioni universali

Le funzioni di NumPy come `np.exp`, `np.sqrt`, `np.maximum` si applicano a ogni elemento dell'array.

In [ ]:
z = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print(np.exp(z))
print(1 / (1 + np.exp(-z)))       # sigmoide su tutti gli elementi
print(np.maximum(z, 0))           # ReLU su tutti gli elementi
print(np.tanh(z))

## 3. Broadcasting

Se le forme sono diverse ma compatibili, NumPy estende l'array più piccolo. Una matrice 3 x 2 più un vettore di 2 elementi: il vettore viene sommato a ogni riga.

In [ ]:
m = np.array([[1, 2],
              [3, 4],
              [5, 6]])
riga = np.array([10, 100])
print(m + riga)

colonna = np.array([[1], [2], [3]])    # forma (3, 1)
print(m * colonna)                      # ogni riga moltiplicata per il proprio valore

## 4. Aggregazioni e parametro axis

`axis=0` opera lungo le righe (un risultato per colonna), `axis=1` lungo le colonne (un risultato per riga).

In [ ]:
voti = np.array([[6, 7, 8],
                 [5, 9, 7],
                 [8, 8, 6],
                 [7, 7, 9]])      # 4 studenti, 3 prove
print("media generale:", voti.mean())
print("media per prova:", voti.mean(axis=0))
print("media per studente:", voti.mean(axis=1))
print("massimo per studente:", voti.max(axis=1))
print("indice del massimo per prova:", voti.argmax(axis=0))

## 5. Confronti e maschere booleane

Un confronto tra un array e un valore produce un array di `True` e `False` (maschera), che si può usare per selezionare elementi.

In [ ]:
t = np.array([18.5, 21.0, 23.4, 19.8, 25.1])
maschera = t > 20
print(maschera)
print(t[maschera])                  # solo i valori maggiori di 20
print(maschera.sum())               # quanti sono: True vale 1, False vale 0
print(np.where(t > 20, 1, 0))       # 1 dove la condizione è vera, 0 altrove

## 6. Numeri casuali

Si crea un generatore con un seme (`seed`): con lo stesso seme si ottengono sempre gli stessi numeri, utile per riprodurre un esperimento.

In [ ]:
rng = np.random.default_rng(42)
print(rng.uniform(0, 1, size=5))            # distribuzione uniforme tra 0 e 1
print(rng.normal(0, 1, size=5))             # distribuzione normale: media 0, deviazione standard 1
print(rng.integers(1, 7, size=10))          # interi da 1 a 6 (7 escluso)

## 7. Velocità: ciclo o operazione vettoriale

`%timeit` è un comando speciale dei notebook (magic command) che esegue un'istruzione più volte e ne misura il tempo medio.

In [ ]:
valori = list(range(100_000))
arr = np.arange(100_000)
%timeit [v * 2 for v in valori]
%timeit arr * 2

## Esercizi

**Esercizio 1 (base).** Scrivere le funzioni `sigmoide(z)` e `relu(z)` che accettano un array e restituiscono l'array dei risultati, senza cicli.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(z, 0)

In [ ]:
zz = np.array([-1.0, 0.0, 2.0])
controlla(1, lambda: np.allclose(sigmoide(zz), [0.26894142, 0.5, 0.88079708]) and np.allclose(relu(zz), [0, 0, 2]))

**Esercizio 2 (base).** Con la matrice `voti` definita sopra calcolare:

- `media_prova_2`: la media della terza colonna
- `migliore`: l'indice dello studente con la media più alta

In [ ]:
media_prova_2 = voti[:, 2].mean()
migliore = voti.mean(axis=1).argmax()

In [ ]:
controlla(2, lambda: np.isclose(media_prova_2, 7.5) and migliore == 3)

**Esercizio 3 (standard).** Normalizzare le misure nell'intervallo [0, 1], senza cicli: `(x - minimo) / (massimo - minimo)`.

In [ ]:
misure = np.array([12.4, 15.1, 9.8, 17.6, 14.0, 11.3, 16.2])
normalizzate = (misure - misure.min()) / (misure.max() - misure.min())
print(normalizzate)

In [ ]:
controlla(3, lambda: normalizzate.min() == 0 and normalizzate.max() == 1 and np.isclose(normalizzate[4], (14.0 - 9.8) / (17.6 - 9.8)))

**Esercizio 4 (standard).** Standardizzazione: sottrarre a ogni colonna la propria media e dividere per la propria deviazione standard (`std(axis=0)`). Il risultato `voti_std` deve avere media 0 e deviazione standard 1 in ogni colonna. Usare il broadcasting.

In [ ]:
voti_std = (voti - voti.mean(axis=0)) / voti.std(axis=0)
print(voti_std)

In [ ]:
controlla(4, lambda: np.allclose(voti_std.mean(axis=0), 0) and np.allclose(voti_std.std(axis=0), 1))

**Esercizio 5 (standard).** Simulare 1000 lanci di un dado con `rng.integers` e calcolare `frequenza_sei`, la frazione di lanci con risultato 6, usando una maschera. Il valore atteso è circa 1/6.

In [ ]:
rng = np.random.default_rng(0)
lanci = rng.integers(1, 7, size=1000)
frequenza_sei = (lanci == 6).mean()
print(frequenza_sei)

In [ ]:
controlla(5, lambda: len(lanci) == 1000 and lanci.min() >= 1 and lanci.max() <= 6 and abs(frequenza_sei - 1/6) < 0.05)

**Esercizio 6 (approfondimento).** Uno strato di neuroni con attivazione a gradino applicato a molti esempi: `Z` contiene le somme pesate (una riga per esempio, una colonna per neurone). Calcolare `uscite`, 1 dove `Z >= 0` e 0 altrove, e `attivi_per_neurone`, il numero di esempi per cui ogni neurone vale 1.

In [ ]:
Z = np.array([[ 0.5, -1.2,  0.0],
              [-0.3,  2.1, -0.7],
              [ 1.4,  0.2, -0.1],
              [-2.0, -0.5,  0.9]])
uscite = np.where(Z >= 0, 1, 0)
attivi_per_neurone = uscite.sum(axis=0)

In [ ]:
controlla(6, lambda: np.array_equal(uscite, [[1, 0, 1], [0, 1, 0], [1, 1, 0], [0, 0, 1]]) and np.array_equal(attivi_per_neurone, [2, 2, 2]))